# 08 Apply inverse to continuous data (labels)

This notebook applies inverse operators to continuous (raw) data and keeps only the vertices of the selected labels. It is meant for analyses on the full time course, for example reverse correlation, where no evoked response or epoching is involved.

Inputs per recording:

- cleaned or filtered raw data,
- inverse operator from `03_inverse_operator.ipynb`,
- labels from the settings in `source.apply_inverse.raw` (parcellation, names, optional merge).

Outputs are written to `derivatives/meeg-pipeline/sub-*/meg/source_estimates_raw/`: one vertex-by-time `.npy` per label, a boolean mask of BAD-annotated samples and a JSON sidecar.

The same step is available from the terminal: `meegpipe source-apply-inverse-raw --config configs/local.yaml --subject <id>`.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import json

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.source_modeling import (
    apply_inverse_to_raw_labels_for_recordings,
    make_raw_label_source_path,
    make_raw_source_bad_mask_path,
    source_estimate_results_to_dataframe,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    selected_recordings_to_dataframe,
    should_overwrite,
)


def find_project_root(start: Path | None = None) -> Path:
    """Find the project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)

## MNE logging

In [ ]:
mne.set_log_level("WARNING")

## Selection

`iter_recordings(..., "all")` uses existing raw-BIDS recordings and excludes `sub-emptyroom` by default.

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

RUN_QC_PLOT = True
QC_PLOT_SECONDS = 30.0

selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)

## Overwrite policy

Default: skip existing raw source estimates.

Set `OVERWRITE_STEPS = ["source_estimate"]` to recompute them.

In [ ]:
OVERWRITE_STEPS = []

source_estimate_policy = existing_output_policy_for_step(
    "source_estimate",
    OVERWRITE_STEPS,
)

pd.DataFrame(
    [
        {
            "step": "source_estimate",
            "overwrite": should_overwrite("source_estimate", OVERWRITE_STEPS),
            "policy": source_estimate_policy,
        }
    ]
)

## Parameters

Methodological parameters are read from `configs/local.yaml` under `source.apply_inverse` and `source.apply_inverse.raw`.

In [ ]:
RAW_SETTINGS = config.source.apply_inverse.raw

METHOD = config.source.apply_inverse.method
SNR = config.source.apply_inverse.snr
LAMBDA2 = config.source.apply_inverse.lambda2
SOURCE_SPACING = config.source.spacing
NOISE_COV_MODE = config.source.noise_cov.mode

pd.DataFrame(
    [
        {
            "method": METHOD,
            "snr": SNR,
            "lambda2": LAMBDA2 if LAMBDA2 is not None else 1.0 / SNR**2,
            "source_spacing": SOURCE_SPACING,
            "noise_cov_mode": NOISE_COV_MODE,
            **RAW_SETTINGS.__dict__,
        }
    ]
).T

## Apply inverse to raw data

This cell processes all selected recordings. Existing results are skipped unless `OVERWRITE_STEPS` contains `"source_estimate"`. Missing inputs and per-recording failures are returned as status rows rather than stopping the whole batch.

In [ ]:
raw_source_results = apply_inverse_to_raw_labels_for_recordings(
    config,
    selected_recordings,
    on_existing=source_estimate_policy,
    method=METHOD,
    lambda2=LAMBDA2,
    snr=SNR,
    spacing=SOURCE_SPACING,
    noise_cov_mode=NOISE_COV_MODE,
    verbose=True,
)

raw_source_results_df = source_estimate_results_to_dataframe(raw_source_results)
raw_source_results_df

## Result summary

In [ ]:
if raw_source_results_df.empty:
    pd.DataFrame()
else:
    (
        raw_source_results_df
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_recordings")
        .sort_values(["status"])
    )

## Sidecar overview

Reads the JSON sidecars of finished recordings: labels, vertex counts, sampling rate and how many samples are excluded by BAD annotations.

In [ ]:
finished = raw_source_results_df[
    raw_source_results_df["status"].isin(["written", "skipped_existing"])
] if not raw_source_results_df.empty else raw_source_results_df

sidecar_rows = []
for _, row in finished.iterrows():
    sidecar = json.loads(Path(row["path"]).read_text(encoding="utf-8"))
    for label in sidecar["labels"]:
        sidecar_rows.append(
            {
                "subject": row["subject"],
                "session": row["session"],
                "task": row["task"],
                "run": row["run"],
                "label": label["name"],
                "n_vertices": label["n_vertices"],
                "sfreq": sidecar["sfreq"],
                "duration_s": sidecar["n_times"] / sidecar["sfreq"],
                "bad_fraction": sidecar["n_bad_samples"] / sidecar["n_times"],
            }
        )

pd.DataFrame(sidecar_rows)

## Visual check

Plots the mean absolute source amplitude per label for the first `QC_PLOT_SECONDS` seconds of each finished recording. Samples inside BAD annotations are shaded.

In [ ]:
if RUN_QC_PLOT and not finished.empty:
    for _, row in finished.iterrows():
        entities = {
            "subject": str(row["subject"]).removeprefix("sub-"),
            "session": row["session"] if pd.notna(row["session"]) else None,
            "task": row["task"] if pd.notna(row["task"]) else None,
            "run": row["run"] if pd.notna(row["run"]) else None,
        }
        sidecar = json.loads(Path(row["path"]).read_text(encoding="utf-8"))
        n_plot = int(QC_PLOT_SECONDS * sidecar["sfreq"])
        bad_mask = np.load(make_raw_source_bad_mask_path(config, **entities))[:n_plot]
        times = np.arange(len(bad_mask)) / sidecar["sfreq"]

        fig, ax = plt.subplots(figsize=(10, 3))
        for label in sidecar["labels"]:
            data = np.load(
                make_raw_label_source_path(
                    config,
                    **entities,
                    label=label["name"],
                    inverse_method=sidecar["method"],
                ),
                mmap_mode="r",
            )[:, :n_plot]
            ax.plot(times, np.abs(data).mean(axis=0), label=label["name"], linewidth=0.8)

        ax.fill_between(
            times,
            0,
            1,
            where=bad_mask,
            transform=ax.get_xaxis_transform(),
            color="0.8",
            label="BAD",
        )
        ax.set(
            title=f"{row['subject']} {row['task'] or ''} {row['run'] or ''}".strip(),
            xlabel="Time (s)",
            ylabel="Mean |source amplitude|",
        )
        ax.legend(loc="upper right", fontsize="small")
        plt.show()
else:
    print("Skipped visual check.")

## Expected outputs

Raw source estimates are written to:

```text
derivatives/meeg-pipeline/sub-*/meg/source_estimates_raw/
```